# 05 — LSTM · Overload Prediction

**Input:** `hive_metastore.gold.gold_features` (pre-scaled, with `split` column)  
**Tracking:** MLflow experiment `Transformer_Overload`

### Notebook structure

| Section | Description |
|---|---|
| 1 | Configuration & imports |
| 2 | Load data & build 96-step sequences |
| 3 | Write sequences to parquet |
| 4 | PyTorch data loader & model definition |
| 5 | Training loop (with early stopping + MLflow) |
| 6 | Evaluation on test set |
| 7 | Threshold sweep |
| 8 | Confusion matrix & curves |
| 9 | Feature importance (permutation) |
| 10 | Summary |

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/03_gold_features/00_Evaluation

## 1 · Configuration & imports

In [0]:
import os, builtins, math, datetime
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pyarrow.dataset as pads
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import mlflow
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
SRC_TABLE    = "hive_metastore.gold.gold_features"
ID_COL       = "ID_prefix"
TS_COL       = "DATE"
LABEL_COL    = "label_4h"            # change to "label_24h" for 24 h horizon
MODEL_NAME   = "LSTM"
EXPERIMENT   = "/Users/daniel.branco@cgi.com/Transformer_Overload"
SEED         = 42

# ── Sequence parameters ──────────────────────────────────────────────────────
N_STEPS = 48    # 48 × 15-min = 12 h look-back window

# ── Feature list ─────────────────────────────────────────────────────────────
CAT_INDEX_COLS = ["ID_prefix_idx", "CONCELHO_idx"]
SIGNAL_COLS = ["current", "voltage"]
LOAD_RATIO_COLS = ["load_ratio_c", "load_ratio_v"]

ROLLING_COLS = [
    f"{s}_{stat}_{w}"
    for s in SIGNAL_COLS
    for stat in ["mean", "std", "max"]
    for w in ["1h", "1d", "7d"]
]

LAG_COLS = [f"{s}_lag_{l}" for s in SIGNAL_COLS for l in ["15m", "1h", "1d"]]

WEATHER_RAW_COLS = [
    "temperatura_media_do_ar_horaria_c",
    "precipitacao_horaria_mm",
    "humidade_relativa_media_horaria_percent",
    "velocidade_do_vento_media_horaria_m_per_s",
]
WEATHER_DERIVED_COLS = ["temp_mean_1d", "temp_mean_7d", "precip_sum_1d"]
TEMPORAL_COLS = ["hour", "day_of_week", "month", "is_weekend"]
EVENT_COLS = ["events_15m_cnt"]

# Numeric features FIRST, then categorical indices LAST
NUMERIC_COLS = (
    LOAD_RATIO_COLS + ROLLING_COLS + LAG_COLS
    + WEATHER_RAW_COLS + WEATHER_DERIVED_COLS
    + TEMPORAL_COLS + EVENT_COLS
)
FEATURE_COLS = NUMERIC_COLS + CAT_INDEX_COLS

N_NUMERIC  = len(NUMERIC_COLS)
INPUT_SIZE = len(FEATURE_COLS)

# ── Training hyperparameters (Galicia et al., 2022; literature-grounded) ────
HIDDEN   = 64       # compact — proportional to dataset size
LAYERS   = 2        # 2 layers for moderate complexity
DROPOUT  = 0.2      # standard starting point (0.2–0.5 range)
LR_RATE  = 0.001     # Adam default, stable for power time-series
EPOCHS   = 15
BATCH_SIZE  = 64
SCAN_BATCH  = 2048
PATIENCE    = 4
VAL_FRAC    = 0.2

# ── Paths ────────────────────────────────────────────────────────────────────
SEQ_DIR   = f"dbfs:/lstm_sequences_{LABEL_COL}"
TRAIN_PATH = f"/dbfs/lstm_sequences_{LABEL_COL}/train"
VAL_PATH   = f"/dbfs/lstm_sequences_{LABEL_COL}/val"
TEST_PATH  = f"/dbfs/lstm_sequences_{LABEL_COL}/test"
SAVE_DIR   = f"/dbfs/models_lstm_{LABEL_COL}"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Sets for classify_feature() ──────────────────────────────────────────────
WEATHER_RAW_SET     = set(WEATHER_RAW_COLS)
WEATHER_DERIVED_SET = set(WEATHER_DERIVED_COLS)
LOAD_RATIO_SET      = set(LOAD_RATIO_COLS)
TEMPORAL_SET        = set(TEMPORAL_COLS)
EVENT_SET           = set(EVENT_COLS)

print(f"Numeric features : {N_NUMERIC}")
print(f"Cat indices      : {CAT_INDEX_COLS}")
print(f"Total per step   : {INPUT_SIZE}")
print(f"Sequence         : {N_STEPS} steps × {INPUT_SIZE} features")
print(f"Architecture     : LSTM({HIDDEN}, {LAYERS}L, drop={DROPOUT}), LR={LR_RATE}")
print(f"Label            : {LABEL_COL}")

## 2 · Load data & build 96-step sequences

Each row in the sequence dataset contains:  
- `features`: array of 96 arrays, each of length `INPUT_SIZE`  
- `label`: the label for the last timestep in the window

Built using `collect_list` over a sliding window, partitioned by transformer.

In [0]:
df = spark.read.table(SRC_TABLE)
print(f"Loaded {df.count():,} rows  |  {len(df.columns)} columns")

# ── Index categoricals (fit on train, apply to all) ──────────────────────
from pyspark.ml.feature import StringIndexer

train_only = df.filter(F.col("split") == "train")
for c in [ID_COL, "CONCELHO"]:
    indexer = StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    df = indexer.fit(train_only).transform(df)

print("Categorical indices added ✅")

# ── Compute cardinalities for embeddings ─────────────────────────────────
id_card   = int(df.select("ID_prefix_idx").agg(F.max("ID_prefix_idx")).first()[0]) + 2
conc_card = int(df.select("CONCELHO_idx").agg(F.max("CONCELHO_idx")).first()[0]) + 2
print(f"ID_prefix cardinality  : {id_card}")
print(f"CONCELHO cardinality   : {conc_card}")

# ── Check for missing features ───────────────────────────────────────────
existing = set(df.columns)
missing = set(FEATURE_COLS) - existing
if missing:
    print(f"⚠️  Missing columns (removed): {missing}")
    FEATURE_COLS[:] = [c for c in FEATURE_COLS if c in existing]
    NUMERIC_COLS[:] = [c for c in NUMERIC_COLS if c in existing]
    N_NUMERIC  = len(NUMERIC_COLS)
    INPUT_SIZE = len(FEATURE_COLS)

# Drop rows with null features or labels
keep = FEATURE_COLS + [LABEL_COL, ID_COL, TS_COL, "split"]
df_clean = df.select([c for c in keep if c in existing]).dropna()
print(f"After dropna: {df_clean.count():,} rows")
print(f"Final features: {len(FEATURE_COLS)} ({N_NUMERIC} numeric + {len(CAT_INDEX_COLS)} cat)")

In [0]:
# Assemble per-row feature array
feat_expr = F.array(*[F.col(c).cast("float") for c in FEATURE_COLS])

dfF = (
    df_clean
    .select(ID_COL, TS_COL, LABEL_COL, "split", feat_expr.alias("f"))
    .withColumn("ts_long", F.col(TS_COL).cast("long"))
)

# Sliding window: collect last 96 rows per transformer
w = Window.partitionBy(ID_COL).orderBy("ts_long").rowsBetween(-N_STEPS + 1, 0)

df_seq = (
    dfF
    .withColumn("features", F.collect_list("f").over(w))
    .where(F.size("features") == N_STEPS)
    .select("features", F.col(LABEL_COL).alias("label"), "split", TS_COL)
)

print(f"Sequences built: {df_seq.count():,}")
df_seq.groupBy("split").count().show()

## 3 · Write sequences to parquet + val split

In [0]:
# Train/val split: last VAL_FRAC of train chronologically → val
train_seqs = df_seq.filter(F.col("split") == "train")
test_seqs  = df_seq.filter(F.col("split") == "test")

date_range = train_seqs.agg(F.min(TS_COL).alias("mn"), F.max(TS_COL).alias("mx")).first()
total_sec = (date_range["mx"] - date_range["mn"]).total_seconds()
val_cutoff = date_range["mn"] + datetime.timedelta(seconds=total_sec * (1 - VAL_FRAC))
print(f"Val cutoff: {val_cutoff}")

train_final = train_seqs.filter(F.col(TS_COL) < F.lit(val_cutoff)).select("features", "label")
val_final   = train_seqs.filter(F.col(TS_COL) >= F.lit(val_cutoff)).select("features", "label")
test_final  = test_seqs.select("features", "label")

print(f"Train seqs : {train_final.count():,}")
print(f"Val seqs   : {val_final.count():,}")
print(f"Test seqs  : {test_final.count():,}")

# Write to parquet
spark.conf.set("spark.sql.files.maxRecordsPerFile", 400000)
train_final.write.mode("overwrite").option("compression", "snappy").parquet(f"{SEQ_DIR}/train")
val_final.write.mode("overwrite").option("compression", "snappy").parquet(f"{SEQ_DIR}/val")
test_final.write.mode("overwrite").option("compression", "snappy").parquet(f"{SEQ_DIR}/test")
print("✅ Sequences written to parquet")

In [0]:
# ── Downsample negatives to make training tractable on CPU ────────────────
NEG_RATIO = 5   # keep 5 negatives per positive

train_spark = spark.read.parquet(f"{SEQ_DIR}/train")
n_pos_seq = train_spark.filter(F.col("label") == 1).count()
n_neg_total = train_spark.filter(F.col("label") == 0).count()
neg_frac = builtins.min(1.0, (n_pos_seq * NEG_RATIO) / builtins.max(n_neg_total, 1))

print(f"Positives : {n_pos_seq:,}")
print(f"Negatives : {n_neg_total:,} → sampling {neg_frac:.4f}")

train_pos = train_spark.filter(F.col("label") == 1)
train_neg = train_spark.filter(F.col("label") == 0).sample(fraction=neg_frac, seed=SEED)
train_ds = train_pos.unionAll(train_neg)

train_ds.write.mode("overwrite").option("compression", "snappy").parquet(f"{SEQ_DIR}/train_ds")

n_ds = train_ds.count()
print(f"Downsampled train: {n_ds:,} sequences (was {n_pos_seq + n_neg_total:,})")

# Update path so training reads the downsampled version
TRAIN_PATH = f"/dbfs/lstm_sequences_{LABEL_COL}/train_ds"

In [0]:
# Compute pos_weight from training sequences
def count_labels(path_local):
    d = pads.dataset(path_local, format="parquet")
    pos = tot = 0
    for frag in d.get_fragments():
        for rec in pads.Scanner.from_fragment(frag, columns=["label"]).to_reader():
            y = np.asarray(rec["label"].to_pylist(), dtype=np.float32)
            pos += int((y == 1).sum()); tot += y.size
    return pos, tot

n_pos, n_total = count_labels(TRAIN_PATH)
n_neg = n_total - n_pos
pos_weight_val = float(n_neg) / builtins.max(n_pos, 1)
print(f"Train: {n_pos:,} pos / {n_total:,} total → pos_weight = {pos_weight_val:.4f}")

## 4 · PyTorch data loader & model definition

In [0]:
def minibatches_from_parquet(folder, batch_size=256, scan_batch=8192, shuffle_files=True):
    """Stream (X, y) minibatches from a parquet folder without loading everything to RAM."""
    dset = pads.dataset(folder, format="parquet")
    fragments = list(dset.get_fragments())
    if shuffle_files:
        np.random.shuffle(fragments)

    bufX, bufy = [], []
    for frag in fragments:
        scanner = pads.Scanner.from_fragment(frag, columns=["features", "label"], batch_size=scan_batch)
        for rec in scanner.to_reader():
            X_list = rec["features"].to_pylist()
            y_list = rec["label"].to_pylist()
            for xi, yi in zip(X_list, y_list):
                bufX.append(xi)
                bufy.append(float(yi))
                if len(bufX) == batch_size:
                    yield np.array(bufX, np.float32), np.array(bufy, np.float32)
                    bufX, bufy = [], []
    if bufX:
        yield np.array(bufX, np.float32), np.array(bufy, np.float32)


print("✅ Data loader ready")

In [0]:
class LSTMWithEmb(nn.Module):
    """
    LSTM with learned embeddings for categorical features.
    Expects input tensor where first N_NUMERIC columns are numeric
    and remaining columns are categorical indices (as floats).
    """
    def __init__(self, n_numeric, emb_specs, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.n_numeric = n_numeric
        self.emb_names = list(emb_specs.keys())
        self.embs = nn.ModuleDict({
            name: nn.Embedding(card, dim)
            for name, (card, dim) in emb_specs.items()
        })
        emb_total = builtins.sum(dim for _, dim in emb_specs.values())
        lstm_input = n_numeric + emb_total

        self.lstm = nn.LSTM(
            lstm_input, hidden, num_layers=layers,
            dropout=(dropout if layers > 1 else 0.0),
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        x_num = x[:, :, :self.n_numeric]
        cat_embs = []
        for i, name in enumerate(self.emb_names):
            idx = x[:, :, self.n_numeric + i].long().clamp(min=0)
            cat_embs.append(self.embs[name](idx))
        if cat_embs:
            x_in = torch.cat([x_num] + cat_embs, dim=-1)
        else:
            x_in = x_num
        _, (hn, _) = self.lstm(x_in)
        return self.head(hn[-1]).squeeze(1)  # logits: (B,)


# ── Embedding specs: (cardinality, embedding_dim) ────────────────────────
EMB_SPECS = {
    "ID_prefix_idx":  (id_card, 8),     # 1000+ IDs → 8 dims
    "CONCELHO_idx":   (conc_card, 4),   # 300+ → 4 dims
}

model = LSTMWithEmb(N_NUMERIC, EMB_SPECS, hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT).to(DEVICE)
opt = optim.AdamW(model.parameters(), lr=LR_RATE, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos_weight_val, device=DEVICE, dtype=torch.float32)
)

n_params = builtins.sum(p.numel() for p in model.parameters())
print(f"Model: {n_params:,} parameters")
print(f"Embeddings: {EMB_SPECS}")
print(f"LSTM input size: {N_NUMERIC} + {builtins.sum(d for _,d in EMB_SPECS.values())} = {N_NUMERIC + builtins.sum(d for _,d in EMB_SPECS.values())}")
print(f"pos_weight = {pos_weight_val:.4f}")
print(model)

## 5 · Training loop (with early stopping + MLflow)

In [0]:
d = pads.dataset(TRAIN_PATH, format="parquet")
n_rows = builtins.sum(frag.count_rows() for frag in d.get_fragments())
print(f"Training on {n_rows:,} sequences")
assert n_rows < 2_000_000, f"Too many rows: {n_rows:,}"

In [0]:
@torch.inference_mode()
def evaluate_epoch(parquet_folder):
    """Evaluate model on a parquet folder, return (y_true, y_prob, avg_loss)."""
    model.eval()
    all_p, all_y, total_loss, n_batches = [], [], 0.0, 0
    for X, y in minibatches_from_parquet(parquet_folder, batch_size=BATCH_SIZE, scan_batch=SCAN_BATCH, shuffle_files=False):
        Xt = torch.from_numpy(X).to(DEVICE)
        yt = torch.from_numpy(y).to(DEVICE)
        logits = model(Xt)
        total_loss += loss_fn(logits, yt).item()
        n_batches += 1
        all_p.append(torch.sigmoid(logits).cpu().numpy())
        all_y.append(y)
    y_true = np.concatenate(all_y)
    y_prob = np.concatenate(all_p)
    avg_loss = total_loss / builtins.max(n_batches, 1)
    return y_true, y_prob, avg_loss

In [0]:
from sklearn.metrics import roc_auc_score, average_precision_score
import gc

mlflow.set_experiment(EXPERIMENT)

best_auprc = -1.0
bad_epochs  = 0
START_EPOCH = 1

# ── Resume from recovery checkpoint if it exists ─────────────────────────
RECOVERY_PATH = os.path.join(SAVE_DIR, "lstm_recovery.pt")
if os.path.exists(RECOVERY_PATH):
    ckpt = torch.load(RECOVERY_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    opt.load_state_dict(ckpt["optimizer_state"])
    best_auprc = ckpt["best_auprc"]
    bad_epochs = ckpt["bad_epochs"]
    START_EPOCH = ckpt["epoch"] + 1
    print(f"✅ Resumed from epoch {ckpt['epoch']}, best_auprc={best_auprc:.4f}")
else:
    print("No recovery checkpoint — starting fresh")

with mlflow.start_run(run_name=f"{MODEL_NAME}_{LABEL_COL}") as run:

    mlflow.log_param("model_type", MODEL_NAME)
    mlflow.log_param("label", LABEL_COL)
    mlflow.log_param("input_size", INPUT_SIZE)
    mlflow.log_param("n_numeric", N_NUMERIC)
    mlflow.log_param("n_steps", N_STEPS)
    mlflow.log_param("hidden", HIDDEN)
    mlflow.log_param("layers", LAYERS)
    mlflow.log_param("dropout", DROPOUT)
    mlflow.log_param("lr", LR_RATE)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("patience", PATIENCE)
    mlflow.log_param("pos_weight", float(f"{pos_weight_val:.4f}"))
    mlflow.log_param("emb_specs", str(EMB_SPECS))

    for epoch in range(START_EPOCH, EPOCHS + 1):
        # ── Train ─────────────────────────────────────────────────────
        model.train()
        train_loss, n_batches = 0.0, 0
        for X, y in minibatches_from_parquet(TRAIN_PATH, BATCH_SIZE, SCAN_BATCH, shuffle_files=True):
            Xt = torch.from_numpy(X).to(DEVICE)
            yt = torch.from_numpy(y).to(DEVICE)
            opt.zero_grad()
            logits = model(Xt)
            loss = loss_fn(logits, yt)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            train_loss += loss.item()
            n_batches += 1

            if n_batches % 50 == 0:
                del Xt, yt, logits, loss
                gc.collect()

        avg_train_loss = train_loss / builtins.max(n_batches, 1)

        # ── Validate ──────────────────────────────────────────────────
        y_val, p_val, avg_val_loss = evaluate_epoch(VAL_PATH)
        val_auroc = float(roc_auc_score(y_val, p_val))
        val_auprc = float(average_precision_score(y_val, p_val))

        mlflow.log_metric("train_loss", avg_train_loss, step=epoch)
        mlflow.log_metric("val_loss", avg_val_loss, step=epoch)
        mlflow.log_metric("val_AUROC", val_auroc, step=epoch)
        mlflow.log_metric("val_AUPRC", val_auprc, step=epoch)

        print(f"Epoch {epoch:02d}  train_loss={avg_train_loss:.4f}  "
              f"val_loss={avg_val_loss:.4f}  val_AUROC={val_auroc:.4f}  val_AUPRC={val_auprc:.4f}")

        # ── Always save recovery checkpoint ───────────────────────────
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": opt.state_dict(),
            "best_auprc": best_auprc,
            "bad_epochs": bad_epochs,
        }, RECOVERY_PATH)
        print(f"  → Recovery checkpoint saved (epoch {epoch})")

        # ── Early stopping on AUPRC ───────────────────────────────────
        if val_auprc > best_auprc:
            best_auprc = val_auprc
            bad_epochs = 0
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, "lstm_best.pt"))
            print(f"  → New best AUPRC={best_auprc:.4f}")
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"  → Early stopping after {PATIENCE} epochs without improvement")
                break

    mlflow.log_metric("best_val_AUPRC", best_auprc)
    best_run_id = run.info.run_id

print(f"\nBest val AUPRC: {best_auprc:.4f}")
print(f"Checkpoint: {os.path.join(SAVE_DIR, 'lstm_best.pt')}")

## 6 · Evaluation on test set

In [0]:
# Load best checkpoint
model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "lstm_best.pt"), map_location=DEVICE))
model.eval()
print("✅ Best checkpoint loaded")

# Evaluate on test
y_true, y_prob, test_loss = evaluate_epoch(TEST_PATH)
print(f"Test loss: {test_loss:.4f}")

final_metrics = evaluate_binary(y_true, y_prob, threshold=0.5, title=f"FINAL TEST — {MODEL_NAME} ({LABEL_COL})")

# Also try fine thresholds
from sklearn.metrics import precision_recall_curve
prec, rec, thrs = precision_recall_curve(y_true, y_prob)
f1_scores = 2 * prec * rec / np.maximum(prec + rec, 1e-8)
best_idx = np.argmax(f1_scores)
print(f"\nPR-curve optimal threshold: {thrs[best_idx]:.6f}")
print(f"F1 at optimal: {f1_scores[best_idx]:.4f}")
print(f"Precision: {prec[best_idx]:.4f}  Recall: {rec[best_idx]:.4f}")

In [0]:
# Log test metrics to MLflow
with mlflow.start_run(run_id=best_run_id):
    log_evaluation_to_mlflow(final_metrics, y_true, y_prob, threshold=0.5, prefix="test")

## 7 · Threshold sweep

In [0]:
sweep_results = threshold_sweep(y_true, y_prob)
display(spark.createDataFrame(sweep_results).orderBy(F.desc("f1")))

In [0]:
best_thr_row = builtins.max(sweep_results, key=lambda r: r["f1"])
optimal_threshold = best_thr_row["threshold"]
print(f"Optimal threshold (max F1): {optimal_threshold}")

optimal_metrics = evaluate_binary(y_true, y_prob, threshold=optimal_threshold, title=f"TEST @ threshold={optimal_threshold} — {MODEL_NAME}")

## 8 · Confusion matrix & curves

In [0]:
fig = plot_confusion_matrix(y_true, (y_prob >= 0.5).astype(int), title=f"{MODEL_NAME} — CM @ 0.5")
display(fig); plt.close(fig)

fig = plot_confusion_matrix(y_true, (y_prob >= optimal_threshold).astype(int), title=f"{MODEL_NAME} — CM @ {optimal_threshold}")
display(fig); plt.close(fig)

In [0]:
fig = plot_roc_curve(y_true, y_prob, title=f"{MODEL_NAME} — ROC")
display(fig); plt.close(fig)

In [0]:
fig = plot_pr_curve(y_true, y_prob, title=f"{MODEL_NAME} — Precision-Recall")
display(fig); plt.close(fig)

## 9 · Feature importance (permutation)

For LSTM, there are no native coefficients or gain.  
We use **permutation importance**: shuffle one feature across all timesteps,  
measure how much AUPRC drops. Bigger drop = more important feature.

In [0]:
from sklearn.metrics import average_precision_score as ap_score

@torch.inference_mode()
def collect_test_data(folder, max_samples=50000):
    """Load a subset of test data into numpy arrays."""
    Xs, ys = [], []
    for X, y in minibatches_from_parquet(folder, batch_size=512, scan_batch=4096, shuffle_files=False):
        Xs.append(X); ys.append(y)
        if builtins.sum(a.shape[0] for a in Xs) >= max_samples:
            break
    return np.concatenate(Xs)[:max_samples], np.concatenate(ys)[:max_samples]


@torch.inference_mode()
def predict_proba(X_np):
    """Get probabilities from numpy array."""
    probs = []
    for i in range(0, X_np.shape[0], 512):
        Xt = torch.from_numpy(X_np[i:i+512]).to(DEVICE)
        probs.append(torch.sigmoid(model(Xt)).cpu().numpy())
    return np.concatenate(probs)


print("Loading test subset for permutation importance...")
X_test, y_test = collect_test_data(TEST_PATH, max_samples=50000)
print(f"Test subset: {X_test.shape}")

# Baseline AUPRC
baseline_probs = predict_proba(X_test)
baseline_auprc = float(ap_score(y_test, baseline_probs))
print(f"Baseline AUPRC: {baseline_auprc:.4f}")

# Permutation importance: shuffle NUMERIC features only (not embeddings)
importances = []
for i, fname in enumerate(NUMERIC_COLS):
    X_perm = X_test.copy()
    perm_idx = np.random.permutation(X_perm.shape[0])
    X_perm[:, :, i] = X_perm[perm_idx, :, i]

    perm_probs = predict_proba(X_perm)
    perm_auprc = float(ap_score(y_test, perm_probs))
    drop = baseline_auprc - perm_auprc
    importances.append(drop)
    if drop > 0.001:
        print(f"  {fname}: AUPRC drop = {drop:.4f}")

importances = np.array(importances)
print(f"\nPermutation importance computed for {len(NUMERIC_COLS)} numeric features")

In [0]:
feat_names = NUMERIC_COLS  # permutation importance only on numeric features

fig_top = plot_top_features(feat_names, importances, top_n=20, title=f"{MODEL_NAME} — Top 20 by Permutation Importance", xlabel="AUPRC Drop")
display(fig_top)

grp = grouped_importance(feat_names, importances)
fig_grp = plot_grouped_importance(grp, title=f"{MODEL_NAME} — Feature Group Importance", xlabel="Sum AUPRC Drop")
display(fig_grp)

# Table
grp_rows = [(g, d["n_dims"], float(d["sum"]), float(d["mean"])) for g, d in grp.items()]
display(spark.createDataFrame(grp_rows, ["group", "n_dims", "sum_auprc_drop", "mean_auprc_drop"]).orderBy(F.desc("sum_auprc_drop")))

In [0]:
import pandas as pd

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_figure(fig_top, "feature_importance_top20.png")
    mlflow.log_figure(fig_grp, "feature_importance_grouped.png")
    mlflow.log_table(
        pd.DataFrame([dict(group=g, **d) for g, d in grp.items()]),
        artifact_file="grouped_feature_importance.json",
    )

plt.close(fig_top)
plt.close(fig_grp)
print("✅ Feature importance logged to MLflow")

## 10 · Summary

In [0]:
print("\n" + "="*60)
print(f"  MODEL SUMMARY — {MODEL_NAME}")
print("="*60)
print(f"  Label          : {LABEL_COL}")
print(f"  Architecture   : LSTMWithEmb({N_NUMERIC}+emb → {HIDDEN}, {LAYERS}L, drop={DROPOUT})")
print(f"  Embeddings     : {EMB_SPECS}")
print(f"  Sequence       : {N_STEPS} steps")
print(f"  pos_weight     : {pos_weight_val:.4f}")
print(f"  LR / Epochs    : {LR_RATE} / {EPOCHS} (patience={PATIENCE})")
print(f"  ─────────────────────────────────")
print(f"  Best val AUPRC : {best_auprc:.4f}")
print(f"  Test AUROC     : {final_metrics['AUROC']:.4f}")
print(f"  Test AUPRC     : {final_metrics['AUPRC']:.4f}")
print(f"  Test F1 @0.5   : {final_metrics['f1']:.4f}")
print(f"  Test MCC @0.5  : {final_metrics['mcc']:.4f}")
print(f"  Optimal thr    : {optimal_threshold}")
print(f"  F1 @opt thr    : {optimal_metrics['f1']:.4f}")
print(f"  ─────────────────────────────────")
print(f"  Checkpoint     : {os.path.join(SAVE_DIR, 'lstm_best.pt')}")
print(f"  MLflow run     : {best_run_id}")
print("="*60)